# Chapter 43 — PyTorch Tensors on CPU

## Learning goals

Earlier chapters used Python lists, NumPy arrays, and homemade scalar autograd.

This chapter introduces the PyTorch tensor object used for vectors, matrices, higher-dimensional arrays, and automatic differentiation.

Every example is explicitly CPU-only.

By the end of this chapter, you should be able to:

1. Create scalar, vector, and matrix tensors.
2. Inspect tensor values, shape, dimension count, dtype, and device.
3. Keep tensor creation and computation on CPU.
4. Perform elementwise arithmetic and matrix multiplication.
5. Create deterministic random tensors.
6. Index tensors and convert small results to Python values.
7. Distinguish floating-point tensors from integer tensors.
8. Explain why token IDs use integer dtype.
9. Use token IDs to select floating-point embedding rows.
10. Explain why gradients flow into embeddings rather than token IDs.
11. Run a small PyTorch autograd example.
12. Explain gradient accumulation and clearing.

## The big idea

A PyTorch tensor is an array-like numerical object with metadata and operations designed for machine learning.

It can represent:

- A scalar loss.
- A vector of features.
- A matrix of weights.
- A batch of token IDs.
- An embedding table.
- Model activations and logits.

Every tensor has values, a shape, a dtype, and a device.

## Terms used in this chapter

- A **scalar tensor** has zero array dimensions and stores one number.
- A **vector tensor** has one dimension.
- A **matrix tensor** has two dimensions.
- **Shape** gives the length along each dimension.
- **Dtype** specifies the numerical representation stored in a tensor.
- **Device** identifies where storage and computation live.
- **CPU** means central processing unit.
- **CUDA** is PyTorch's NVIDIA GPU platform, which this course does not use.
- A **token ID** is an integer vocabulary index.
- An **embedding table** stores one floating-point vector per token ID.
- A **differentiable tensor** participates in autograd and may receive a gradient.

PyTorch autograd supports floating-point and complex tensors, while this course focuses on floating-point values.

## Import PyTorch and choose CPU

Set the device explicitly without checking for CUDA or moving anything to a GPU.

In [1]:
import torch

device = "cpu"

print("PyTorch version:", torch.__version__)
print("Course device:", device)

assert device == "cpu"

PyTorch version: 2.2.2
Course device: cpu


## Create the required vector tensor

Explicit dtype removes any dependence on the process-wide default floating dtype.

In [2]:
numbers = torch.tensor(
    [1.0, 2.0, 3.0],
    dtype=torch.float32,
    device=device,
)

print("Values:", numbers)
print("Shape:", numbers.shape)
print("Dimensions:", numbers.ndim)
print("Number of elements:", numbers.numel())
print("Dtype:", numbers.dtype)
print("Device:", numbers.device)

assert numbers.shape == torch.Size([3])
assert numbers.ndim == 1
assert numbers.numel() == 3
assert numbers.dtype == torch.float32
assert numbers.device.type == "cpu"

Values: tensor([1., 2., 3.])
Shape: torch.Size([3])
Dimensions: 1
Number of elements: 3
Dtype: torch.float32
Device: cpu


## Scalar, vector, and matrix shapes

A scalar tensor has shape `[]`, a length-three vector has shape `[3]`, and a two-by-two matrix has shape `[2, 2]`.

The number of dimensions is also called the tensor's rank in many machine-learning discussions.

In [3]:
scalar_tensor = torch.tensor(7.0, dtype=torch.float32, device=device)
vector_tensor = torch.tensor(
    [1.0, 2.0, 3.0],
    dtype=torch.float32,
    device=device,
)
matrix_tensor = torch.tensor(
    [
        [1.0, 2.0],
        [3.0, 4.0],
    ],
    dtype=torch.float32,
    device=device,
)

print("kind   | shape | ndim | values")
print("-" * 49)
print(
    f"scalar | {str(scalar_tensor.shape):>13} | "
    f"{scalar_tensor.ndim:>4} | {scalar_tensor}"
)
print(
    f"vector | {str(vector_tensor.shape):>13} | "
    f"{vector_tensor.ndim:>4} | {vector_tensor}"
)
print(
    f"matrix | {str(matrix_tensor.shape):>13} | "
    f"{matrix_tensor.ndim:>4} |\n{matrix_tensor}"
)

assert scalar_tensor.shape == torch.Size([])
assert vector_tensor.shape == torch.Size([3])
assert matrix_tensor.shape == torch.Size([2, 2])

kind   | shape | ndim | values
-------------------------------------------------
scalar | torch.Size([]) |    0 | 7.0
vector | torch.Size([3]) |    1 | tensor([1., 2., 3.])
matrix | torch.Size([2, 2]) |    2 |
tensor([[1., 2.],
        [3., 4.]])


## Shape gives structure

Language-model tensors commonly use shapes such as:

```text
token IDs:   [batch size, context length]
embeddings:  [batch size, context length, embedding width]
logits:      [batch size, context length, vocabulary size]
targets:     [batch size, context length]
```

The same stored numbers can mean different things under a different shape, so shape is part of the model contract.

## Floating-point and integer dtypes

`torch.float32` is common for learned numerical values, while `torch.long` is PyTorch's alias for signed 64-bit integers and is common for IDs and labels.

In [4]:
floating_values = torch.tensor(
    [1.0, 2.0, 3.0],
    dtype=torch.float32,
    device=device,
)
integer_values = torch.tensor(
    [1, 2, 3],
    dtype=torch.long,
    device=device,
)

print("Floating-point tensor:", floating_values)
print("Floating-point dtype:", floating_values.dtype)
print("Integer tensor:", integer_values)
print("Integer dtype:", integer_values.dtype)
print("torch.long equals torch.int64?", torch.long == torch.int64)

assert floating_values.dtype == torch.float32
assert integer_values.dtype == torch.int64
assert floating_values.device.type == integer_values.device.type == "cpu"

Floating-point tensor: tensor([1., 2., 3.])
Floating-point dtype: torch.float32
Integer tensor: tensor([1, 2, 3])
Integer dtype: torch.int64
torch.long equals torch.int64? True


## Different dtypes serve different roles

Floating-point tensors commonly store parameters, embeddings, activations, logits, probabilities, and losses.

Integer tensors commonly store token IDs, class IDs, positions, and other indexes.

An ID's numerical magnitude is not a measurement: token ID 10 is not “twice as much token” as token ID 5.

## Basic elementwise arithmetic

Tensors with the same shape can be added and multiplied element by element.

In [5]:
first_values = torch.tensor(
    [1.0, 2.0, 3.0],
    dtype=torch.float32,
    device=device,
)
second_values = torch.tensor(
    [10.0, 20.0, 30.0],
    dtype=torch.float32,
    device=device,
)

elementwise_sum = first_values + second_values
elementwise_product = first_values * second_values

print("First tensor:", first_values)
print("Second tensor:", second_values)
print("Elementwise sum:", elementwise_sum)
print("Elementwise product:", elementwise_product)

assert torch.equal(elementwise_sum, torch.tensor([11.0, 22.0, 33.0]))
assert torch.equal(elementwise_product, torch.tensor([10.0, 40.0, 90.0]))
assert elementwise_sum.device.type == elementwise_product.device.type == "cpu"

First tensor: tensor([1., 2., 3.])
Second tensor: tensor([10., 20., 30.])
Elementwise sum: tensor([11., 22., 33.])
Elementwise product: tensor([10., 40., 90.])


## Scalar operations and broadcasting

Adding one scalar to a vector applies that scalar to every element.

This is a simple example of broadcasting, where PyTorch aligns compatible shapes automatically.

In [6]:
offset_vector = numbers + 10.0
doubled_vector = numbers * 2.0

print("Original:", numbers)
print("Add scalar 10:", offset_vector)
print("Multiply by scalar 2:", doubled_vector)

assert torch.equal(offset_vector, torch.tensor([11.0, 12.0, 13.0]))
assert torch.equal(doubled_vector, torch.tensor([2.0, 4.0, 6.0]))

Original: tensor([1., 2., 3.])
Add scalar 10: tensor([11., 12., 13.])
Multiply by scalar 2: tensor([2., 4., 6.])


## Matrix-vector multiplication

The `@` operator performs matrix multiplication rather than elementwise multiplication.

In [7]:
matrix_for_vector = torch.tensor(
    [
        [1.0, 2.0],
        [3.0, 4.0],
    ],
    dtype=torch.float32,
    device=device,
)
vector_for_matrix = torch.tensor(
    [10.0, 20.0],
    dtype=torch.float32,
    device=device,
)
matrix_vector_result = matrix_for_vector @ vector_for_matrix

print("Matrix:")
print(matrix_for_vector)
print("Vector:", vector_for_matrix)
print("Matrix @ vector:", matrix_vector_result)
print("Result shape:", matrix_vector_result.shape)

assert torch.equal(matrix_vector_result, torch.tensor([50.0, 110.0]))
assert matrix_vector_result.shape == torch.Size([2])

Matrix:
tensor([[1., 2.],
        [3., 4.]])
Vector: tensor([10., 20.])
Matrix @ vector: tensor([ 50., 110.])
Result shape: torch.Size([2])


## Matrix-matrix multiplication

A `[2, 3]` matrix can multiply a `[3, 2]` matrix because the inner dimensions match, producing shape `[2, 2]`.

In [8]:
left_matrix = torch.tensor(
    [
        [1.0, 2.0, 3.0],
        [4.0, 5.0, 6.0],
    ],
    dtype=torch.float32,
    device=device,
)
right_matrix = torch.tensor(
    [
        [10.0, 20.0],
        [30.0, 40.0],
        [50.0, 60.0],
    ],
    dtype=torch.float32,
    device=device,
)
matrix_matrix_result = left_matrix @ right_matrix

print("Left shape:", left_matrix.shape)
print("Right shape:", right_matrix.shape)
print("Result:")
print(matrix_matrix_result)
print("Result shape:", matrix_matrix_result.shape)

expected_matrix_product = torch.tensor(
    [[220.0, 280.0], [490.0, 640.0]],
    dtype=torch.float32,
)
assert torch.equal(matrix_matrix_result, expected_matrix_product)
assert matrix_matrix_result.shape == torch.Size([2, 2])

Left shape: torch.Size([2, 3])
Right shape: torch.Size([3, 2])
Result:
tensor([[220., 280.],
        [490., 640.]])
Result shape: torch.Size([2, 2])


## Create zeros and ones

Tensor factories accept a shape, dtype, and device without first constructing nested Python lists.

In [9]:
zero_vector = torch.zeros(5, dtype=torch.float32, device=device)
one_matrix = torch.ones((2, 3), dtype=torch.float32, device=device)

print("Zero vector:", zero_vector)
print("Zero-vector shape:", zero_vector.shape)
print("One matrix:")
print(one_matrix)
print("One-matrix shape:", one_matrix.shape)

assert zero_vector.shape == torch.Size([5])
assert one_matrix.shape == torch.Size([2, 3])
assert torch.count_nonzero(zero_vector).item() == 0
assert one_matrix.sum().item() == 6.0

Zero vector: tensor([0., 0., 0., 0., 0.])
Zero-vector shape: torch.Size([5])
One matrix:
tensor([[1., 1., 1.],
        [1., 1., 1.]])
One-matrix shape: torch.Size([2, 3])


## Deterministic random tensors

An isolated CPU generator makes the random seed explicit without changing unrelated global random state.

Recreating a generator with the same seed reproduces the same values in this locked environment.

In [10]:
RANDOM_SEED = 43

first_generator = torch.Generator(device=device)
first_generator.manual_seed(RANDOM_SEED)
first_random_matrix = torch.randn(
    (2, 3),
    generator=first_generator,
    dtype=torch.float32,
    device=device,
)

second_generator = torch.Generator(device=device)
second_generator.manual_seed(RANDOM_SEED)
second_random_matrix = torch.randn(
    (2, 3),
    generator=second_generator,
    dtype=torch.float32,
    device=device,
)

print("First random matrix:")
print(first_random_matrix)
print("Second random matrix:")
print(second_random_matrix)
print("Equal?", torch.equal(first_random_matrix, second_random_matrix))

assert torch.equal(first_random_matrix, second_random_matrix)
assert first_random_matrix.device.type == "cpu"

First random matrix:
tensor([[-0.6484, -0.7058,  0.6432],
        [ 1.4788,  1.1918, -0.1446]])
Second random matrix:
tensor([[-0.6484, -0.7058,  0.6432],
        [ 1.4788,  1.1918, -0.1446]])
Equal? True


## Indexing and Python conversion

Indexing a vector once returns a zero-dimensional tensor, and `.item()` converts that one-element tensor to a Python scalar.

`.tolist()` is convenient for small displays but is not how large tensor computations should be performed.

In [11]:
first_number_tensor = numbers[0]
first_number_python = first_number_tensor.item()
matrix_entry_tensor = matrix_tensor[0, 1]

print("numbers[0]:", first_number_tensor)
print("numbers[0] shape:", first_number_tensor.shape)
print("numbers[0].item():", first_number_python)
print("matrix[0, 1]:", matrix_entry_tensor)
print("numbers.tolist():", numbers.tolist())

assert first_number_tensor.shape == torch.Size([])
assert first_number_python == 1.0
assert matrix_entry_tensor.item() == 2.0

numbers[0]: tensor(1.)
numbers[0] shape: torch.Size([])
numbers[0].item(): 1.0
matrix[0, 1]: tensor(2.)
numbers.tolist(): [1.0, 2.0, 3.0]


## Token IDs form integer tensors

This fake batch has two sequences and four positions per sequence, so its shape is `[2, 4]`.

In [12]:
token_ids = torch.tensor(
    [
        [5, 8, 2, 9],
        [5, 7, 3, 4],
    ],
    dtype=torch.long,
    device=device,
)

print("Token IDs:")
print(token_ids)
print("Shape:", token_ids.shape)
print("Dtype:", token_ids.dtype)
print("Device:", token_ids.device)

assert token_ids.shape == torch.Size([2, 4])
assert token_ids.dtype == torch.long
assert token_ids.device.type == "cpu"

Token IDs:
tensor([[5, 8, 2, 9],
        [5, 7, 3, 4]])
Shape: torch.Size([2, 4])
Dtype: torch.int64
Device: cpu


## Token IDs are lookup keys

An ID selects one row from an embedding table.

The integer ID is a label, while the selected row contains continuous floating-point features that a model can learn.

In [13]:
embedding_table = torch.tensor(
    [
        [0.10, 0.20, 0.30],
        [0.40, 0.50, 0.60],
        [0.70, 0.80, 0.90],
        [1.00, 1.10, 1.20],
    ],
    dtype=torch.float32,
    device=device,
)
lookup_token_ids = torch.tensor(
    [0, 2, 1, 3],
    dtype=torch.long,
    device=device,
)
selected_embeddings = embedding_table[lookup_token_ids]

print("Embedding-table shape:", embedding_table.shape)
print("Token IDs:", lookup_token_ids)
print("Selected embeddings:")
print(selected_embeddings)
print("Selected shape:", selected_embeddings.shape)

assert selected_embeddings.shape == torch.Size([4, 3])
assert torch.equal(selected_embeddings[1], embedding_table[2])

Embedding-table shape: torch.Size([4, 3])
Token IDs: tensor([0, 2, 1, 3])
Selected embeddings:
tensor([[0.1000, 0.2000, 0.3000],
        [0.7000, 0.8000, 0.9000],
        [0.4000, 0.5000, 0.6000],
        [1.0000, 1.1000, 1.2000]])
Selected shape: torch.Size([4, 3])


## Integer tensors cannot require ordinary gradients

PyTorch rejects `requires_grad=True` for an integer tensor because derivatives describe infinitesimal changes, while integer IDs are discrete lookup choices.

Catch the expected error so the notebook continues executing.

In [14]:
try:
    torch.tensor(
        [1, 2, 3],
        dtype=torch.long,
        device=device,
        requires_grad=True,
    )
except RuntimeError as integer_gradient_error:
    print("Integer gradient request failed as expected:")
    print(integer_gradient_error)
else:
    raise AssertionError("An integer tensor should not accept requires_grad=True.")

Integer gradient request failed as expected:
Only Tensors of floating point and complex dtype can require gradients


## Gradients flow into the embedding table

Make the floating-point table differentiable, use integer IDs to select rows, and sum the selected values into a scalar loss.

Repeated ID 2 selects the same row twice, so that row receives twice the gradient contribution.

In [15]:
learnable_embedding_table = torch.tensor(
    [
        [0.10, 0.20, 0.30],
        [0.40, 0.50, 0.60],
        [0.70, 0.80, 0.90],
        [1.00, 1.10, 1.20],
    ],
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
repeated_token_ids = torch.tensor(
    [0, 2, 1, 2],
    dtype=torch.long,
    device=device,
)
learned_lookup = learnable_embedding_table[repeated_token_ids]
lookup_loss = learned_lookup.sum()
lookup_loss.backward()  # type: ignore[no-untyped-call]

print("Token IDs:", repeated_token_ids)
print("Embedding-table gradient:")
print(learnable_embedding_table.grad)
print("Token IDs require gradients?", repeated_token_ids.requires_grad)
print("Token-ID gradient attribute:", repeated_token_ids.grad)

expected_embedding_gradient = torch.tensor(
    [
        [1.0, 1.0, 1.0],
        [1.0, 1.0, 1.0],
        [2.0, 2.0, 2.0],
        [0.0, 0.0, 0.0],
    ],
    dtype=torch.float32,
)
assert learnable_embedding_table.grad is not None
assert torch.equal(learnable_embedding_table.grad, expected_embedding_gradient)
assert not repeated_token_ids.requires_grad
assert repeated_token_ids.grad is None

Token IDs: tensor([0, 2, 1, 2])
Embedding-table gradient:
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [2., 2., 2.],
        [0., 0., 0.]])
Token IDs require gradients? False
Token-ID gradient attribute: None


## Floating-point tensors can participate in autograd

`requires_grad=True` tells PyTorch to track operations needed to differentiate a final scalar with respect to this leaf tensor.

Not every floating-point input needs gradients; model parameters usually do, while ordinary data often does not.

In [16]:
weight = torch.tensor(
    1.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
input_number = torch.tensor(5.0, dtype=torch.float32, device=device)
target_number = torch.tensor(10.0, dtype=torch.float32, device=device)

print("Gradient before backward:", weight.grad)

prediction = weight * input_number
squared_error_loss = (prediction - target_number) ** 2
squared_error_loss.backward()

print("Prediction:", prediction)
print("Loss:", squared_error_loss)
print("Weight gradient:", weight.grad)

assert weight.grad is not None
assert weight.grad.item() == -50.0
assert prediction.device.type == squared_error_loss.device.type == "cpu"

Gradient before backward: None
Prediction: tensor(5., grad_fn=<MulBackward0>)
Loss: tensor(25., grad_fn=<PowBackward0>)
Weight gradient: tensor(-50.)


## PyTorch gradients accumulate

Recompute the forward graph and call backward again without clearing `weight.grad`.

The same `-50` contribution is added, producing `-100`.

In [17]:
second_prediction = weight * input_number
second_loss = (second_prediction - target_number) ** 2
second_loss.backward()

print("Gradient after a second backward pass:", weight.grad)

assert weight.grad is not None
assert weight.grad.item() == -100.0

Gradient after a second backward pass: tensor(-100.)


## Clear gradients before the next step

Setting a leaf gradient to `None` is a simple manual reset for this example.

Optimizers later provide `zero_grad()` for clearing all model parameter gradients together.

In [18]:
weight.grad = None

third_prediction = weight * input_number
third_loss = (third_prediction - target_number) ** 2
third_loss.backward()

print("Gradient after clearing and recomputing:", weight.grad)

assert weight.grad is not None
assert weight.grad.item() == -50.0

Gradient after clearing and recomputing: tensor(-50.)


## Homemade autograd and PyTorch autograd

The earlier `TrackedNumber` object stored scalar data, gradients, graph parents, and backward rules.

PyTorch applies the same computational-graph idea to whole tensors and provides optimized implementations of tensor operations.

This chapter is only a preview; the next chapter will use PyTorch autograd in complete parameter-update loops.

## CPU-only checklist

Use the explicit course device for tensor factories:

```python
device = "cpu"
values = torch.tensor([1.0, 2.0, 3.0], device=device)
zeros = torch.zeros((2, 3), device=device)
```

When combining tensors, keep them on the same device.

This course intentionally avoids hardware detection, CUDA branches, and GPU-only assumptions.

## A clean tensor-inspection pipeline

For every important tensor, inspect values, shape, dtype, and device before relying on it in a larger computation.

In [19]:
inspection_vector = torch.tensor(
    [1.0, 2.0, 3.0],
    dtype=torch.float32,
    device=device,
)
inspection_matrix = torch.tensor(
    [[1.0, 2.0], [3.0, 4.0]],
    dtype=torch.float32,
    device=device,
)

for tensor_name, inspected_tensor in (
    ("vector", inspection_vector),
    ("matrix", inspection_matrix),
):
    print(tensor_name)
    print("  values:", inspected_tensor)
    print("  shape:", inspected_tensor.shape)
    print("  dtype:", inspected_tensor.dtype)
    print("  device:", inspected_tensor.device)

vector
  values: tensor([1., 2., 3.])
  shape: torch.Size([3])
  dtype: torch.float32
  device: cpu
matrix
  values: tensor([[1., 2.],
        [3., 4.]])
  shape: torch.Size([2, 2])
  dtype: torch.float32
  device: cpu


## What not to do

- Do not write code that assumes CUDA in this CPU-only course.
- Do not ignore shape, dtype, or device when debugging tensor code.
- Do not confuse elementwise `*` with matrix multiplication `@`.
- Do not represent token IDs as floating-point measurements.
- Do not try to differentiate integer token IDs.
- Do not conclude that embeddings are nondifferentiable merely because their lookup IDs are integers.
- Do not forget that leaf gradients are `None` before backward and accumulate after repeated backward passes.
- Do not compare random values across unlocked environments and assume a seed guarantees identical implementation details forever.

## Gotchas

### Shape is part of meaning

`[batch, context]` token IDs and `[batch, context, embedding]` vectors have different roles.

### Dtype controls valid operations

Floating-point tensors support ordinary gradient tracking, while integer tensors serve as discrete indexes and labels.

### Device consistency matters

All tensors in these examples live on CPU and report `tensor.device.type == "cpu"`.

### Lookup and learning are separate

Integer IDs choose embedding rows, and floating-point embedding values receive gradients.

### Gradients accumulate

Clear old parameter gradients before computing gradients for a new training step.

## Takeaways

A PyTorch tensor combines numerical values with shape, dtype, device, and a large operation library.

This course explicitly uses:

```python
device = "cpu"
```

Floating-point tensors store parameters, embeddings, activations, logits, and losses.

Integer tensors store token IDs and class IDs used for lookup and indexing.

The key distinction is:

```text
integer token IDs select rows
floating-point embedding rows receive gradients
```

PyTorch autograd extends the computational-graph ideas from homemade scalar autograd to entire tensors.

## What comes next

The next chapter uses PyTorch autograd to train one-parameter and two-parameter models.

The familiar loop remains:

```text
forward pass → loss → backward pass → parameter update → clear gradients
```